In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
import pickle
import os

df = pd.read_csv('../data/processed/final_dataset.csv')
print(f'Shape: {df.shape}')

Shape: (30000, 36)


In [12]:
x = df.drop(columns=['url', 'label', 'domain_age_days', 'domain_registration_length'])
y = df['label']

print(f'Features: {x.shape[1]}')
print(f'Samples: {x.shape[0]}')
print(f'Feature names: {list(x.columns)}')

Features: 32
Samples: 30000
Feature names: ['url_length', 'hostname_length', 'nb_dots', 'nb_hyphens', 'nb_slash', 'nb_at', 'nb_qmark', 'nb_equal', 'nb_underscore', 'nb_percent', 'nb_digits', 'ratio_digits_url', 'ratio_digits_host', 'https_token', 'ip_in_hostname', 'nb_subdomains', 'prefix_suffix', 'suspicious_tld', 'shortening_service', 'phishing_hints', 'brand_similarity', 'path_length', 'has_port', 'double_slash_in_path', 'dns_record', 'nb_hyperlinks', 'ratio_internal_links', 'nb_external_css', 'has_login_form', 'has_iframe', 'external_favicon', 'title_matches_domain']


In [15]:
for col in x.columns:
    if (x[col] == -1).any():
        median_val = x[col][x[col] != -1].median()
        x[col] = x[col].replace(-1, median_val)
        print(f"Imputed {col} with median {median_val:.2f}")

print("\nMissing after imputation:", x.isnull().sum().sum())


Missing after imputation: 0


In [17]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {x_train.shape} | test: {x_test.shape}')
print(f'Train balance: {y_train.value_counts()}')
print(f'Test balance: {y_test.value_counts()}')

Train: (24000, 32) | test: (6000, 32)
Train balance: label
0    12000
1    12000
Name: count, dtype: int64
Test balance: label
0    3000
1    3000
Name: count, dtype: int64


In [18]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=min(30, x_train.shape[1])))
])

x_train_processed = pipeline.fit_transform(x_train, y_train)
x_test_processed = pipeline.transform(x_test)

print(f'After selection: {x_train_processed.shape[1]} features')

with open('../models/feature_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

np.save('../models/x_train.npy', x_train_processed)
np.save('../models/x_test.npy', x_test_processed)
np.save('../models/y_train.npy', y_train.values)
np.save('../models/y_test.npy', y_test.values)

with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(list(x.columns), f)

print('Pipeline and splits saved')

After selection: 30 features
Pipeline and splits saved


c:\Project\Phish-Print-v2\venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [24] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Project\Phish-Print-v2\venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
